# ML Pipeline Demo - Traffic Flow Prediction

This notebook demonstrates training ML models to predict traffic flows using SUMO simulation data.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from traffic_control import (
    generate_four_junction_example,
    generate_manhattan_example,
    run_ml_pipeline,
    predict_flows,
    TrafficFlowRegressor,
)

## Generate Network and Run ML Pipeline

Note: This requires SUMO to be installed. If SUMO is not available, the pipeline will fail.

In [ ]:
network = generate_manhattan_example()

try:
    models = run_ml_pipeline(
        network,
        output_dir="../models",
        model_types=["linear", "ridge", "random_forest"],
        duration=3600,
        lookback=4,
    )
    print("Models trained successfully!")
except Exception as e:
    print(f"Pipeline failed (likely SUMO not installed): {e}")

## Evaluate Trained Models

In [ ]:
import joblib
from pathlib import Path

model_dir = Path("../models")
if model_dir.exists():
    for model_file in model_dir.glob("*.joblib"):
        data = joblib.load(model_file)
        print(f"{model_file.stem}: MAE={data.get('metrics', {}).get('mae', 'N/A'):.4f}")
else:
    print("No models found. Run the pipeline first.")

## Make Predictions

In [ ]:
try:
    predictions = predict_flows(
        network,
        model_id="random_forest",
        features={
            "hour": 8,
            "time_of_day": 0.5,
            "length": 200,
            "capacity": 1000,
            "cost": 1.0,
            "utilization": 0.3,
            "travel_time": 120,
        },
        model_dir="../models",
    )
    print("Predictions:", predictions)
except Exception as e:
    print(f"Prediction failed: {e}")